## Práctica Desempeño: Implementación del algoritmo Naive Bayes (Gaussian Naive Bayes)

## Dataset: Wine (incluido en scikit-learn)

**Carga del dataset**

from sklearn.datasets import load_wine

# Contexto del Problema

El dataset Wine contiene el resultado de un análisis químico de vinos cultivados en la misma región de Italia, pero provenientes de **tres cultivos distintos**.

Cada registro describe un vino mediante 13 mediciones químicas (alcohol, magnesio, flavonoides, etc.). El objetivo será construir un modelo capaz de predecir a cuál de los tres cultivos pertenece un vino a partir de esas mediciones:

- class_0
- class_1
- class_2

El interés práctico de este problema es evidente en el sector vitivinícola: contar con un modelo que, a partir de un análisis de laboratorio, prediga el origen o cultivo de un vino permite automatizar controles de calidad, detectar posibles mezclas o etiquetados incorrectos, y apoyar decisiones de clasificación cuando el origen no es evidente a simple vista.



In [ ]:
#Importamos librerias
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
accuracy_score,
confusion_matrix,
classification_report,
ConfusionMatrixDisplay
)
import seaborn as sns
import matplotlib.pyplot as plt


In [ ]:
#Cargamos el dataset Wine
from sklearn.datasets import load_wine
wine = load_wine(as_frame = True)


In [ ]:
print (wine.DESCR)


In [ ]:
dfWine = wine.frame
dfWine.head()


## Exploracion del conjunto de datos


In [ ]:
dfWine.shape


In [ ]:
dfWine.columns


In [ ]:
dfWine.info()


In [ ]:
dfWine.describe()


In [ ]:
wine.target_names


In [ ]:
#Verificamos valores nulos
dfWine.isnull().sum()


In [ ]:
### Renombrar la columna "target"

dfWine.rename(columns = {"target" : "cultivo"}, inplace = True)


In [ ]:
dfWine["cultivo"] = dfWine["cultivo"].map({
    0: "class_0",
    1: "class_1",
    2: "class_2"
})


In [ ]:
dfWine.head()


In [ ]:
dfWine.tail()


In [ ]:
#Distribucion de la variable objetivo
dfWine["cultivo"].value_counts()


In [ ]:
#Graficamos la distribucion
dfWine["cultivo"].value_counts().plot(
    kind = "bar"
)
plt.title("Distribucion de cultivos")
plt.show()


## Identificar variables predictoras y variables objetivos


In [ ]:
#Variable predictora
X = dfWine.drop(columns = ["cultivo"])


In [ ]:
print("Variables predictoras")
print(X.columns)


In [ ]:
#Varible objetivo
y = dfWine["cultivo"]


In [ ]:
print("Variable objetivo")
print(y.name)


## Dividir los datos

80% para entrenamiento

20% para prueba


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size= 0.20, random_state= 42, stratify=y
)


In [ ]:
print("X_train", X_train.shape)
print("X_test", X_test.shape)
print("y_train", y_train.shape)
print("y_test", y_test.shape)


## Entrenamiento del modelo GaussianNB



In [ ]:
modelo = GaussianNB()


In [ ]:
modelo.fit(
    X_train, y_train
)


## Realizamos predicciones con las variables de Prueba


In [ ]:
prediccion = modelo.predict(X_test)


In [ ]:
print (prediccion[:10])


In [ ]:
#Comparación entre valores reales y valores predecibles
comparacion = pd.DataFrame({
    "cultivo real": y_test.values,
    "cultivo predicho": prediccion
})


In [ ]:
comparacion.head(10)


## Evaluación del modelo


In [ ]:
accuracy = accuracy_score(y_test, prediccion)
print(f"accuracy/exactitud: {accuracy: .4f}")


In [ ]:
## Matriz de confusion
cm = confusion_matrix(y_test, prediccion, labels=modelo.classes_)


In [ ]:
cm_df = pd.DataFrame(
    cm,
    index= modelo.classes_,
    columns= modelo.classes_
)
cm_df


In [ ]:
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=modelo.classes_)
disp.plot(cmap="Blues")
plt.title("Matriz de confusion - Naive Bayes")
plt.show()


## Clasificacion report
Proporciona métricas que permitan evaluar el desempeño de cada clase: precission, recall, f1-score, support


In [ ]:
print(classification_report(y_test, prediccion))


## Importancia de las variables (Permutation Importance)

`GaussianNB` **no tiene un atributo `feature_importances_`** como sí lo tienen los modelos basados en árboles: no divide el espacio por variable, por lo que no existe una importancia "nativa" que extraer directamente del modelo.

Para obtener una medida de importancia válida para cualquier modelo (incluyendo Naive Bayes) se usa **Permutation Importance**: la idea es tomar el conjunto de prueba ya evaluado, revolver (permutar) aleatoriamente los valores de **una sola variable a la vez** manteniendo las demás intactas, y medir cuánto **cae el accuracy**. Si una variable es importante para el modelo, revolverla destruye su relación con la clase real y el accuracy baja notablemente; si una variable casi no la usa el modelo, revolverla apenas afecta el resultado.


In [ ]:
perm = permutation_importance(
    modelo, X_test, y_test,
    n_repeats=30, random_state=42
)

importancia_df = pd.DataFrame({
    "variable": X.columns,
    "importancia_media": perm.importances_mean,
    "importancia_std": perm.importances_std
}).sort_values("importancia_media", ascending=False)

importancia_df


In [ ]:
plt.figure(figsize=(8,6))
plt.barh(
    importancia_df["variable"],
    importancia_df["importancia_media"],
    xerr=importancia_df["importancia_std"]
)
plt.gca().invert_yaxis()
plt.xlabel("Caida en accuracy al permutar la variable")
plt.title("Importancia de variables (Permutation Importance) - Naive Bayes")
plt.tight_layout()
plt.show()


## Verificación del efecto del escalado

Se entrena una segunda versión del modelo, esta vez sobre los datos escalados con `StandardScaler` (ajustado solo con `X_train` para evitar fuga de información), para comprobar de forma empírica que GaussianNB no se beneficia del escalado.


In [ ]:
scaler = StandardScaler()
X_train_escalado = scaler.fit_transform(X_train)
X_test_escalado = scaler.transform(X_test)

modelo_escalado = GaussianNB()
modelo_escalado.fit(X_train_escalado, y_train)
prediccion_escalada = modelo_escalado.predict(X_test_escalado)

accuracy_escalado = accuracy_score(y_test, prediccion_escalada)

print(f"Accuracy sin escalar: {accuracy: .4f}")
print(f"Accuracy escalado:    {accuracy_escalado: .4f}")
print("Predicciones identicas en ambos casos:", (prediccion == prediccion_escalada).all())


## Implementación con datos nuevos

Se construye un vino **ficticio**, fuera del conjunto de entrenamiento y de prueba, con valores químicos plausibles (basados en los rangos típicos observados por clase con `dfWine.groupby("cultivo").mean()`). No se aplica ningún escalado adicional porque, como se demostró arriba, GaussianNB no lo requiere: el nuevo dato se pasa directamente al modelo en las mismas 13 columnas y unidades que `X_train`.


In [ ]:
#Vino ficticio, con valores tipicos de la clase class_1
nuevo_vino = pd.DataFrame([{
    "alcohol": 12.3,
    "malic_acid": 1.9,
    "ash": 2.2,
    "alcalinity_of_ash": 20.0,
    "magnesium": 100.0,
    "total_phenols": 2.3,
    "flavanoids": 2.1,
    "nonflavanoid_phenols": 0.35,
    "proanthocyanins": 1.5,
    "color_intensity": 3.5,
    "hue": 1.05,
    "od280/od315_of_diluted_wines": 2.8,
    "proline": 520.0
}])
nuevo_vino


In [ ]:
prediccion_nueva = modelo.predict(nuevo_vino)
probabilidades_nueva = modelo.predict_proba(nuevo_vino)

print("Cultivo predicho:", prediccion_nueva[0])
print()
print("Probabilidad por clase:")
for clase, prob in zip(modelo.classes_, probabilidades_nueva[0]):
    print(f"  {clase}: {prob:.6f}")


## Conclusiones

**Resultado obtenido:** el modelo GaussianNB alcanzó un **accuracy de 0.9722** sobre el conjunto de prueba (35 de 36 vinos clasificados correctamente), lo que indica un desempeño alto para este problema de clasificación con tres clases.

**¿Dónde falló el modelo?** La matriz de confusión muestra un único error: un vino de `class_1` fue clasificado como `class_0` (precision de `class_0` = 0.92, recall de `class_1` = 0.93). El resto de los vinos de prueba fueron clasificados correctamente en las tres clases.

**El escalado no tuvo ningún efecto.** Entrenar el modelo con y sin `StandardScaler` produjo **exactamente las mismas predicciones y el mismo accuracy**. Esto confirma en la práctica lo que predice la teoría: GaussianNB estima media y varianza por variable de forma independiente, y una transformación afín (como estandarizar) no cambia cuál clase obtiene mayor probabilidad.

**Importancia de variables:** el permutation importance identificó a `flavanoids`, `color_intensity` y `proline` como las variables que más caída de accuracy provocan al ser permutadas, es decir, las que más información aportan al modelo para separar los tres cultivos. Varias variables (`magnesium`, `ash`, `proanthocyanins`) tuvieron importancia cercana a cero o incluso ligeramente negativa (ruido estadístico esperable con solo 36 registros de prueba), lo que sugiere que un subconjunto más pequeño de variables podría alcanzar un desempeño similar.

**Limitaciones — por qué no hay que sobreinterpretar el 0.9722:**
- El conjunto de prueba es pequeño (36 registros): un solo vino mal clasificado mueve el accuracy 2.8 puntos porcentuales, así que el resultado debe leerse con cautela y no como una medida perfectamente estable del desempeño real del modelo.
- El supuesto de independencia entre variables no se cumple realmente en este dataset (por ejemplo, `flavanoids` y `total_phenols` están correlacionados químicamente); el modelo funcionó bien a pesar de eso, pero no hay garantía de que ese comportamiento se sostenga en un dataset con variables más correlacionadas entre sí.
- No se validó formalmente el supuesto de normalidad por variable y por clase (por ejemplo con un histograma o una prueba de Shapiro-Wilk); si alguna variable está muy sesgada dentro de una clase, `GaussianNB` está modelando su distribución de forma incorrecta aunque el resultado final se vea bien.
- Para una evaluación más robusta, lo recomendable sería usar **validación cruzada** sobre el conjunto de entrenamiento en lugar de una sola partición train/test, ya que un solo split de 36 datos de prueba no es suficiente para concluir con total confianza qué tan bien generaliza el modelo.

**Aprendizaje técnico:** esta práctica mostró que Naive Bayes es un algoritmo simple, rápido y sorprendentemente competitivo cuando las variables aportan información discriminante razonable, aunque el supuesto de independencia no se cumpla estrictamente. Su salida probabilística (`predict_proba`) es una ventaja frente a modelos que solo entregan una etiqueta, ya que permite comunicar el nivel de confianza de cada predicción.
